### K-Nearest Neighbours

In [43]:
from scipy.sparse import load_npz, csr_matrix
import numpy as np
import pandas as pd

#### 1) Carga de matrices: 

In [44]:
# 1) Cargamos las matrices de usuarios-items, tanto de train como de test: 
path = '../data/processed/'

train_set = load_npz(path+'train_set.npz')
test_set = load_npz(path+'test_set.npz')

#### 2) Cálculo de la similitud: 

In [45]:
# 2) Definimos una función para calcular similitud entre usuarios. 
# Aplicaremos la similitud coseno por nuestro conocimiento sobre él al haberla usado en PLN
# y debido a sus buenos resultados: 

def cosine_similarity_user(user_idx, matrix):
    """
    Calcula similitud coseno entre un usuario y todos los demás.
    """
    # Extraemos vector del usuario: 
    user_vec = matrix[user_idx]
    
    # Calculamos norma L2 del usuario: 
    user_norm = np.sqrt(np.array(user_vec.power(2).sum()))
    if user_norm == 0:
        user_norm = 1.0
    
    # Calculamos normas L2 de todos los usuarios: 
    all_norms = np.sqrt(np.array(matrix.power(2).sum(axis=1)).flatten())
    all_norms[all_norms == 0] = 1.0
    
    # Aplicamos el producto escalar de un usuario con todos los demás de forma vectorizada: 
    dot_products = (matrix @ user_vec.T).toarray().flatten()
    
    # Dividimos por normas: 
    similarities = dot_products / (user_norm * all_norms)
    
    return similarities

#### 3) Obtención de vecinos: 

In [46]:
# 3) Definimos una función para obtener los k vecinos más similares con reespecto a un usuario: 

def get_k_neighbours(user_idx, matrix, k=10):
    """
    Encuentra los k vecinos más similares a un usuario.
    """
    # Calculamos similitudes coseno usando la función creada previamente: 
    similarities = cosine_similarity_user(user_idx, matrix)
    
    # Ignoramos similitud consigo mismo: 
    similarities[user_idx] = -1.0
    
    # Encontramos los k índices con mayor similitud usando argpartition, 
    # que permite encontrar estos k índices sin ordenar todo: 
    top_k_indices = np.argpartition(similarities, -k)[-k:]
    top_k_similarities = similarities[top_k_indices]
    
    # Ordenamos descendentemente solo los k elementos: 
    sort_order = np.argsort(top_k_similarities)[::-1]
    
    return top_k_indices[sort_order], top_k_similarities[sort_order]


# Prueba de funcionamiento:
vecinos_ids, similitudes = get_k_neighbours(user_idx=150, matrix=train_set, k=10)
vecinos_ids, similitudes

(array([51628, 42456, 26989,  3962, 11133, 14063, 25520, 49436, 65499,
        40727]),
 array([0.16881607, 0.16574797, 0.15663457, 0.15661834, 0.15398313,
        0.14752863, 0.13800162, 0.13377616, 0.13282748, 0.13258311],
       dtype=float32))

#### 4) Estimación de las predicciones: 

In [47]:
# 4) Definimos una función para obtener la predicción dee la valoración de un usuario sobre un ítem i a partir de sus vecinos: 

def mean_users(users, matrix): 
    """
    Calcula la media de valoraciones/playcounts de uno o varios usuarios.
    """

    # Convertimos a lista/array por si entra un escalar (un solo usuario):
    users = np.atleast_1d(users)
    rows = matrix[users]

    row_sums = np.asarray(rows.sum(axis=1)).ravel()
    row_counts = np.diff(rows.indptr)

    # División segura para evitar warnings si un usuario tiene 0 escuchas.
    # Si row_counts es 0, asigna 0 como media por defecto:
    means = np.divide(row_sums, row_counts, out=np.zeros_like(row_sums, dtype=float), where=row_counts!=0)

    return means

def deviation_from_mean_prediction(u, i, neighbours, matrix): 
    """
    Función vectorizada que calcula la predicción de la valoración de un ítem i 
    por parte de un usuario u según sus k vecinos.
    """
    # 1) Extraemos la columna del ítem i solo para los vecinos indicados
    sub_matrix = matrix[neighbours, i]
    
    # Obtenemos los índices relativos de los que sí votaron el ítem:
    relevant_idx = sub_matrix.nonzero()[0]
    n = relevant_idx.size

    # Si ningún vecino votó el ítem, no podemos predecir:
    if n == 0:
        return None

    # Mapeamos los índices relativos a los IDs reales de los vecinos:
    actual_relevant_neighbours = neighbours[relevant_idx]

    # 2) Obtenemos las medias de los vecinos relevantes y la media del usuario u:
    relevant_neighbours_means = mean_users(actual_relevant_neighbours, matrix)
    u_mean = mean_users(u, matrix)[0] # Extraemos el escalar del array resultante

    # 3) Calculamos la fórmula final de la desviación.
    # Extraemos los ratings reales aislando a los vecinos válidos: 
    ratings_i = matrix[actual_relevant_neighbours, i].toarray().ravel()
    
    deviation = (ratings_i - relevant_neighbours_means).sum() / n
    prediction = u_mean + deviation

    return prediction

def user_prediction(u, matrix, k=20):
    """
    Calcula las predicciones de ítems para un usuario u basándose en sus k vecinos.
    """

    num_items = matrix.shape[1]
    
    # 1) Inicializamos el array de predicciones con -infinito.
    # Esto asegura que los ítems no calculados o ya vistos sean ignorados posteriormente: 
    predictions = np.full(num_items, -np.inf)
    
    # 2) Obtenemos los vecinos usando la función optimizada anterior: 
    neighbors, _ = get_k_neighbours(u, matrix, k=k)
    
    if len(neighbors) == 0:
        return predictions # Si no hay vecinos, devolvemos -inf para todo.

    # 3) Optimizamos el espacio de búsqueda: 

    # 3.1) Obtenemos todos los ítems que han escuchado los vecinos.
    neighbours_items = np.unique(matrix[neighbors].indices) # .indices devuelve todas las columnas distintas de cero. 
    
    # 3.2) Obtenemos los ítems que el usuario u ya ha escuchado:
    user_seen_items = matrix[u].indices
    
    # 3.3) Nos quedamos solo con los ítems de los vecinos que el usuario no conoce (np.setdiff1d):
    items_to_predict = np.setdiff1d(neighbours_items, user_seen_items)
    
    # 4) Calculamos la predicción solo para ese pequeño grupo de candidatos (máxima eficiencia): 
    for i in items_to_predict:

        # Llamamos a la función previa: 
        pred = deviation_from_mean_prediction(u, i, neighbors, matrix)
        
        # Si la función nos devuelve un valor válido, lo guardamos: 
        if pred is not None:
            predictions[i] = pred
            
    return predictions

#### 5) Cálculo de las recomendaciones: 

In [48]:
def get_recommendations(predictions, N=5): 
    ''' 
    Función que devuelve las k mejores recomendaciones según las estimaciones predichas. 
    '''

    # 1) Aseguramos que trabajamos con un array de NumPy: 
    preds_array = np.asarray(predictions, dtype=float)
    
    # 2) Manejo de valores nulos por -infinito: 
    preds_array[np.isnan(preds_array)] = -np.inf
    
    # 3) Control de seguridad:
    valid_count = np.sum(preds_array != -np.inf)
    actual_n = min(N, valid_count)
    
    if actual_n == 0:
        return np.array([]), np.array([]) # No hay nada que recomendar.
        
    # 4) Extraer los índices de las N predicciones más altas.
    # argpartition coloca los 'actual_n' mayores al final, pero sin orden interno:
    top_n_indices = np.argpartition(preds_array, -actual_n)[-actual_n:]
    
    # Extraemos los valores correspondientes a esos índices: 
    top_n_values = preds_array[top_n_indices]
    
    # 5) Ordenar estrictamente ese pequeño subconjunto de mayor a menor: 
    sort_order = np.argsort(top_n_values)[::-1]
    
    final_items = top_n_indices[sort_order]
    final_scores = top_n_values[sort_order]
    
    return final_items, final_scores

#### 6) Modelo unificado de KNN: 

In [ ]:
class KNN_model(): 

    def __init__(self, train_set: csr_matrix, test_set, k=20): 
        self.train_set = train_set
        self.test_set = test_set
        self.k = k
    
    def recommend(self, user_id, n_recommendations=5): 
        ''' 
        Genera predicciones y recomendaciones para un usuario específico.
        '''
        # Calculamos predicciones para todos los ítems usando la función user_prediction: 
        predictions = user_prediction(user_id, self.train_set, k=self.k)

        # Obtenemos las top-N recomendaciones usando get_recommendations: 
        recommended_items, scores = get_recommendations(predictions, N=n_recommendations)
        
        return recommended_items, scores
    
    def evaluate(self):
        ''' 
        Evalúa el modelo usando el conjunto de test y la función predefinida en metrics.py.
        '''
        import sys
        import os
        sys.path.append(os.path.abspath('../src'))
        from metrics import evaluate_model
        
        # Convertimos test_set a DataFrame en caso de haber instanciado con una CSR matrix:
        if not isinstance(self.test_set, pd.DataFrame):
            coo = self.test_set.tocoo()
            test_df = pd.DataFrame({
                'user_n': coo.row,
                'track_n': coo.col,
                'rating': coo.data
            })
        else:
            test_df = self.test_set.copy()
            
        test_df['pred'] = 0.0

        # Iteramos por cada usuario en test para no generar recomendaciones de más:
        for user_id in test_df['user_n'].unique():

            # Usamos la función de predicción sobre el usuario correspondiente: 
            preds_all = user_prediction(user_id, self.train_set, k=self.k)
            
            # Obtenemos los ítems que ese usuario en específico tiene registrados en test: 
            mask = test_df['user_n'] == user_id
            track_ids = test_df.loc[mask, 'track_n'].values
            
            # Extraemos de las predicciones aquellas canciones de ese usuario que están en test: 
            user_preds = preds_all[track_ids]
            
            # En caso de que no haya vecinos que puntuaron (-inf), imputamos con 0 a modo de penalización: 
            user_preds[np.isinf(user_preds)] = 0.0
            test_df.loc[mask, 'pred'] = user_preds

        # Llamamos a nuestra función pasándole el dataframe estructurado y los array predichos:
        return evaluate_model(test_df, test_df['pred'].values)

In [51]:
# Comprobación de su funcionamiento: 
knn_model = KNN_model(train_set, test_set)
knn_model.recommend(3)

(array([11579,  1564,  2891, 16163,  5279]),
 array([5.27963639, 5.21599657, 5.14152319, 5.01566531, 4.9387864 ]))

In [52]:
# Evaluación según métricas: 
knn_model.evaluate()

ValueError: Unable to coerce to Series, length must be 4: given 1930594